# Exploratory Data Analysis (EDA) - Cleaned Data

This notebook performs exploratory data analysis on the cleaned startups dataset produced by the data cleaning and preprocessing pipeline. The main supervised-learning target is the binary variable `success`.

In [ ]:
# Imports and setup

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

print("Imports complete.")


## 1. Load cleaned dataset

We work from the cleaned dataset saved by the preprocessing notebook (`02_data_cleaning_and_preprocessing.ipynb`). This CSV already contains engineered features and the main target variable `success`.

In [ ]:
# Load the cleaned dataset

DATA_PATH = "../data/processed/startups_clean.csv"
df = pd.read_csv(DATA_PATH)

print("Cleaned dataset loaded from:", DATA_PATH)
print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print("\nColumns:")
print(list(df.columns))

display(df.head())


## 2. Target definition: `success`

The main target used for supervised learning in this project is the binary variable `success`, which has already been computed in the cleaned dataset.

- `years_alive` is the number of years between the founding date and the last observed date for the startup (approximated by the latest available funding date when an explicit closing or scrape date is not present).
- `survived_5y` is a boolean flag equal to `True` if `years_alive ≥ 5`.
- `success` is defined as:
  - `success = 1` if `years_alive ≥ 5` **or** `status` is in `{\"acquired\", \"ipo\"}`
  - `success = 0` otherwise

In this notebook we **reuse** the `success` column from the cleaned CSV and do not recompute it from scratch.

### 2.1 Distribution of `success`

We first inspect the distribution of the target variable `success` (0 = not successful, 1 = successful).

In [ ]:
# Distribution of the success target

if "success" in df.columns:
    print("Value counts for success:")
    print("=" * 60)
    counts = df["success"].value_counts(dropna=False).sort_index()
    pct = df["success"].value_counts(normalize=True, dropna=False).sort_index() * 100

    summary_df = pd.DataFrame({
        "Count": counts,
        "Percentage": pct.round(2)
    })
    display(summary_df)

    plt.figure(figsize=(6, 5))
    counts.plot(kind="bar", color=["salmon", "seagreen"], edgecolor="black")
    plt.title("Distribution of Success", fontsize=14, fontweight="bold")
    plt.xlabel("success (0 = no, 1 = yes)")
    plt.ylabel("Count")
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()
else:
    print("Column 'success' not found in the cleaned dataset.")


### 2.2 Relationship between `success` and key features

We now explore how `success` relates to a few important features already present in the cleaned dataset (e.g., total funding and region). These are exploratory visualizations to build intuition and are not final modeling features.

In [ ]:
# Relationship between success and total funding

if {"success", "funding_total_usd"}.issubset(df.columns):
    df["funding_total_usd_num"] = pd.to_numeric(df["funding_total_usd"], errors="coerce")

    plt.figure(figsize=(10, 6))
    sns.boxplot(
        data=df.dropna(subset=["funding_total_usd_num", "success"]),
        x="success",
        y="funding_total_usd_num"
    )
    plt.yscale("log")
    plt.title("Funding vs. Success (log scale)", fontsize=14, fontweight="bold")
    plt.xlabel("success (0 = no, 1 = yes)")
    plt.ylabel("Total funding (USD, log scale)")
    plt.tight_layout()
    plt.show()
else:
    print("Required columns for funding vs success plot not available.")

# Relationship between success rate and region (top regions)
if {"success", "region"}.issubset(df.columns):
    region_success = (
        df.dropna(subset=["region", "success"])
        .groupby("region")["success"]
        .mean()
        .sort_values(ascending=False)
        .head(15)
    )

    plt.figure(figsize=(12, 6))
    region_success.plot(kind="bar", color="steelblue", edgecolor="black")
    plt.title("Success Rate by Region (Top 15)", fontsize=14, fontweight="bold")
    plt.xlabel("Region")
    plt.ylabel("Mean success rate")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()
else:
    print("Required columns for region vs success plot not available.")


## 3. Notes

- All analyses in this notebook are based on the cleaned dataset `startups_clean.csv` produced by the preprocessing pipeline.
- The primary target for supervised learning is the binary variable `success` defined as: 1 if a startup either survives at least 5 years or has an exit event (`acquired` or `ipo`), and 0 otherwise.
- Additional descriptive plots of features such as `status`, funding, geography, etc. can be added on top of this foundation without changing the target definition.